In [1]:
import urllib.request
import zipfile
import os
from pathlib import Path

url = "https://archive.ics.uci.edu/static/public/228/sms+spam+collection.zip"
zip_path= "sms_spam_collection.zip"
extracted_path= "sms_spam_collection"
data_file_path= Path(extracted_path)  / "SMSSpamCollection.tsv"

def download_and_unzip_spam_data(
        url, zip_path, extracted_path, data_file_path):
    if data_file_path.exists():  
        print(f"{data_file_path} already exists. Skipping download"
              "and extraction."
              )
        return

    
    with urllib.request.urlopen(url) as response:
        with open(zip_path, "wb" ) as out_file:
            out_file.write(response.read())

    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(extracted_path)

    original_file_path= Path(extracted_path)/ "SMSSpamCollection"
    os.rename(original_file_path, data_file_path)
    print(f"file donwloaded and saved as {data_file_path}")

download_and_unzip_spam_data(url, zip_path, extracted_path, data_file_path)




sms_spam_collection\SMSSpamCollection.tsv already exists. Skipping downloadand extraction.


In [2]:
import pandas as pd
df= pd.read_csv(
    data_file_path, sep= "\t", header= None, names= ["Label", "Text"]
)
df

,Label,Text
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."
...,...,...
5567,spam,This is the 2nd time we have tried 2 contact u...
5568,ham,Will ü b going to esplanade fr home?
5569,ham,"Pity, * was in mood for that. So...any other s..."
5570,ham,The guy did some bitching but I acted like i'd...


In [3]:
print(df["Label"].value_counts())

Label
ham     4825
spam     747
Name: count, dtype: int64


In [4]:

print( df[df["Label"]== "spam"]  )

     Label                                               Text
2     spam  Free entry in 2 a wkly comp to win FA Cup fina...
5     spam  FreeMsg Hey there darling it's been 3 week's n...
8     spam  WINNER!! As a valued network customer you have...
9     spam  Had your mobile 11 months or more? U R entitle...
11    spam  SIX chances to win CASH! From 100 to 20,000 po...
...    ...                                                ...
5537  spam  Want explicit SEX in 30 secs? Ring 02073162414...
5540  spam  ASKED 3MOBILE IF 0870 CHATLINES INCLU IN FREE ...
5547  spam  Had your contract mobile 11 Mnths? Latest Moto...
5566  spam  REMINDER FROM O2: To get 2.50 pounds free call...
5567  spam  This is the 2nd time we have tried 2 contact u...

[747 rows x 2 columns]


In [5]:
def create_balanced_dataset(df):
    num_spam= df[df["Label"]== "spam"].shape[0]
    ham_subset= df[df["Label"]== "ham"].sample(
        num_spam, random_state=123
    )

    balanced_df = pd.concat([
        ham_subset, df[df["Label"]== "spam"]
    ])
    return balanced_df

balanced_df= create_balanced_dataset(df)
print(balanced_df["Label"].value_counts())

Label
ham     747
spam    747
Name: count, dtype: int64


In [6]:
balanced_df["Label"]= balanced_df["Label"].map({"ham":0, "spam":1})

In [7]:
def random_split(df, train_frac, validation_frac):

    df= df.sample(
        frac=1, random_state= 123
    ).reset_index(drop = True)
    train_end = int(len(df) * train_frac)
    validation_end = train_end + int(len(df) * validation_frac)

    train_df = df[: train_end]
    validation_df= df[train_end: validation_end]

    test_df= df[validation_end:]

    return train_df, validation_df, test_df

train_df, validation_df, test_df= random_split(
    balanced_df, 0.7, 0.1
)

In [8]:
train_df.to_csv("train.csv", index= None)
validation_df.to_csv("validation.csv", index= None)
test_df.to_csv("test.csv", index= None)

In [9]:
import tiktoken
tokenizer= tiktoken.get_encoding("gpt2")
print(tokenizer.encode("<|endoftext|>", allowed_special= {"<|endoftext|>"}))

[50256]


In [10]:
print(tokenizer.decode([50256]))

<|endoftext|>


In [11]:
print(tokenizer.decode([91, 437, 1659, 5239, 91]))

|endoftext|


dataset class

In [12]:
import torch
import pandas as pd
from torch.utils.data import Dataset


class SpamDataset(Dataset):
    def __init__(self, csv_file, tokenizer, max_length=None, pad_token_id=50256):
        self.data = pd.read_csv(csv_file)

        # 1. Encode all texts
        self.encoded_texts = [
            tokenizer.encode(text)
            for text in self.data["Text"]
        ]

        # Determine maximum sequence length
        if max_length is None:
            self.max_length = self.longest_encoded_length()
        else:
            self.max_length = max_length

        # 2. Truncate sequences
        self.encoded_texts = [
            encoded_text[:self.max_length]
            for encoded_text in self.encoded_texts
        ]

        # 3. Pad sequences
        self.encoded_texts = [
            encoded_text + [pad_token_id] * (self.max_length - len(encoded_text))
            for encoded_text in self.encoded_texts
        ]

    def __getitem__(self, index):
        encoded = self.encoded_texts[index]
        label = self.data.iloc[index]["Label"]

        return (
            torch.tensor(encoded, dtype=torch.long),
            torch.tensor(label, dtype=torch.long),
        )

    def __len__(self):
        return len(self.data)

    def longest_encoded_length(self):
        max_length = 0

        for encoded_text in self.encoded_texts:
            encoded_length = len(encoded_text)
            if encoded_length > max_length:
                max_length = encoded_length

        return max_length

In [13]:
data= pd.read_csv("train.csv")
data

,Label,Text
0,0,Dude how do you like the buff wind.
1,0,Tessy..pls do me a favor. Pls convey my birthd...
2,1,Reminder: You have not downloaded the content ...
3,1,Got what it takes 2 take part in the WRC Rally...
4,1,"Shop till u Drop, IS IT YOU, either 10K, 5K, £..."
...,...,...
1040,1,4mths half price Orange line rental & latest c...
1041,1,Thanks for the Vote. Now sing along with the s...
1042,1,IMPORTANT INFORMATION 4 ORANGE USER 0796XXXXXX...
1043,1,Urgent! call 09066612661 from landline. Your c...


In [14]:
for i, text in enumerate(data["Text"]):
    print(i, text , "\n")

    if(i>10):
        break
    

0 Dude how do you like the buff wind. 

1 Tessy..pls do me a favor. Pls convey my birthday wishes to Nimya..pls dnt forget it. Today is her birthday Shijas 

2 Reminder: You have not downloaded the content you have already paid for. Goto http://doit. mymoby. tv/ to collect your content. 

3 Got what it takes 2 take part in the WRC Rally in Oz? U can with Lucozade Energy! Text RALLY LE to 61200 (25p), see packs or lucozade.co.uk/wrc & itcould be u! 

4 Shop till u Drop, IS IT YOU, either 10K, 5K, £500 Cash or £100 Travel voucher, Call now, 09064011000. NTT PO Box CR01327BT fixedline Cost 150ppm mobile vary 

5 Am on a train back from northampton so i'm afraid not! I'm staying skyving off today ho ho! Will be around wednesday though. Do you fancy the comedy club this week by the way? 

6 Dude. What's up. How Teresa. Hope you have been okay. When i didnt hear from these people, i called them and they had received the package since dec  &lt;#&gt; . Just thot you'ld like to know. Do have a 

In [15]:
train_dataset= SpamDataset(
    csv_file= "train.csv",
    max_length= None,
    tokenizer= tokenizer
)

In [16]:
print(train_dataset.max_length )

120


In [17]:
val_dataset= SpamDataset(
    csv_file = "validation.csv",
    max_length = train_dataset.max_length,
    tokenizer= tokenizer
)

test_dataset = SpamDataset(
    csv_file= "test.csv",
    max_length= train_dataset.max_length,
    tokenizer= tokenizer
)


In [18]:
# dataloader
import torch
from torch.utils.data import DataLoader
num_workers=0
batch_size= 8
torch.manual_seed(123)

train_loader = DataLoader(
    dataset= train_dataset,
    batch_size= batch_size,
    shuffle = True,
    num_workers= num_workers,
    drop_last= True
)

val_loader= DataLoader(
    dataset= val_dataset,
    batch_size= batch_size,
    num_workers= num_workers,
    drop_last= False
)

test_loader= DataLoader(
    dataset= test_dataset,
    batch_size= batch_size,
    num_workers= num_workers,
    drop_last= False  
)

In [19]:
for input_batch, target_batch in train_loader:
    pass
print("input batch dimensions :", input_batch.shape)
print("label batch dimensions :", target_batch.shape)

input batch dimensions : torch.Size([8, 120])
label batch dimensions : torch.Size([8])


In [20]:
print(f"{len(train_loader)} training batches")
print(f"{len(val_loader)} validation batches")

130 training batches
19 validation batches


In [21]:
CHOOSE_MODEL = "gpt2-small (124M)"
INPUT_PROMPT = "Every effort moves"

BASE_CONFIG = {
    "vocab_size": 50257,      #1
    "context_length": 1024,   #2
    "drop_rate": 0.0,         #3
    "qkv_bias": True          #4
}

model_configs = {
    "gpt2-small (124M)": {
        "emb_dim": 768,
        "n_layers": 12,
        "n_heads": 12
    },
    "gpt2-medium (355M)": {
        "emb_dim": 1024,
        "n_layers": 24,
        "n_heads": 16
    },
    "gpt2-large (774M)": {
        "emb_dim": 1280,
        "n_layers": 36,
        "n_heads": 20
    },
    "gpt2-xl (1558M)": {
        "emb_dim": 1600,
        "n_layers": 48,
        "n_heads": 25
    },
}

BASE_CONFIG.update(model_configs[CHOOSE_MODEL])

In [23]:
from gpt_download import download_and_load_gpt2
from previous_chapters import GPTModel, load_weights_into_gpt

model_size= CHOOSE_MODEL.split(" ")[-1].strip("(").rstrip(")")
settings, params= download_and_load_gpt2(
    model_size= model_size, models_dir= "gpt2"
)

model= GPTModel(BASE_CONFIG)
load_weights_into_gpt(model, params)
model.eval()

File already exists and is up-to-date: gpt2\124M\checkpoint
File already exists and is up-to-date: gpt2\124M\encoder.json
File already exists and is up-to-date: gpt2\124M\hparams.json
File already exists and is up-to-date: gpt2\124M\model.ckpt.data-00000-of-00001
File already exists and is up-to-date: gpt2\124M\model.ckpt.index
File already exists and is up-to-date: gpt2\124M\model.ckpt.meta
File already exists and is up-to-date: gpt2\124M\vocab.bpe


GPTModel(
  (tok_emb): Embedding(50257, 768)
  (pos_emb): Embedding(1024, 768)
  (drop_emb): Dropout(p=0.0, inplace=False)
  (trf_blocks): Sequential(
    (0): TransformerBlock(
      (att): MultiHeadAttention(
        (W_query): Linear(in_features=768, out_features=768, bias=True)
        (W_key): Linear(in_features=768, out_features=768, bias=True)
        (W_value): Linear(in_features=768, out_features=768, bias=True)
        (out_proj): Linear(in_features=768, out_features=768, bias=True)
        (dropout): Dropout(p=0.0, inplace=False)
      )
      (ff): FeedForward(
        (layers): Sequential(
          (0): Linear(in_features=768, out_features=3072, bias=True)
          (1): GELU()
          (2): Linear(in_features=3072, out_features=768, bias=True)
        )
      )
      (norm1): LayerNorm()
      (norm2): LayerNorm()
      (drop_resid): Dropout(p=0.0, inplace=False)
    )
    (1): TransformerBlock(
      (att): MultiHeadAttention(
        (W_query): Linear(in_features=768,

In [30]:
from previous_chapters import generate_text_simple, text_to_token_ids, token_ids_to_text

text_1= "jai"
token_ids = generate_text_simple(
    model= model,
    idx= text_to_token_ids(text_1, tokenizer),
    max_new_tokens= 115,
    context_size= BASE_CONFIG["context_length"]
)

print(token_ids_to_text(token_ids, tokenizer))

jai, a former member of the U.S. Army's Special Operations Command, said the U.S. military has been "very clear" that it will not use force against the Islamic State.

"We are not going to use force against the Islamic State," he said. "We are not going to use force against the Islamic State. We are not going to use force against the Islamic State."

The U.S. military has been in the midst of a major campaign to defeat the Islamic State, which has taken over large parts of Iraq and


In [31]:
print(model)

GPTModel(
  (tok_emb): Embedding(50257, 768)
  (pos_emb): Embedding(1024, 768)
  (drop_emb): Dropout(p=0.0, inplace=False)
  (trf_blocks): Sequential(
    (0): TransformerBlock(
      (att): MultiHeadAttention(
        (W_query): Linear(in_features=768, out_features=768, bias=True)
        (W_key): Linear(in_features=768, out_features=768, bias=True)
        (W_value): Linear(in_features=768, out_features=768, bias=True)
        (out_proj): Linear(in_features=768, out_features=768, bias=True)
        (dropout): Dropout(p=0.0, inplace=False)
      )
      (ff): FeedForward(
        (layers): Sequential(
          (0): Linear(in_features=768, out_features=3072, bias=True)
          (1): GELU()
          (2): Linear(in_features=3072, out_features=768, bias=True)
        )
      )
      (norm1): LayerNorm()
      (norm2): LayerNorm()
      (drop_resid): Dropout(p=0.0, inplace=False)
    )
    (1): TransformerBlock(
      (att): MultiHeadAttention(
        (W_query): Linear(in_features=768,

In [32]:
for param in model.parameters():
    param.requires_grad= False

In [33]:
torch.manual_seed(123)
num_classes=2
model.out_head = torch.nn.Linear(
    in_features= BASE_CONFIG["emb_dim"],
    out_features= num_classes
)


In [34]:
for param in model.trf_blocks[-1].parameters():
    param.requires_grad= True
for param in model.final_norm.parameters():
    param.requires_grad= True

In [35]:
print(model)

GPTModel(
  (tok_emb): Embedding(50257, 768)
  (pos_emb): Embedding(1024, 768)
  (drop_emb): Dropout(p=0.0, inplace=False)
  (trf_blocks): Sequential(
    (0): TransformerBlock(
      (att): MultiHeadAttention(
        (W_query): Linear(in_features=768, out_features=768, bias=True)
        (W_key): Linear(in_features=768, out_features=768, bias=True)
        (W_value): Linear(in_features=768, out_features=768, bias=True)
        (out_proj): Linear(in_features=768, out_features=768, bias=True)
        (dropout): Dropout(p=0.0, inplace=False)
      )
      (ff): FeedForward(
        (layers): Sequential(
          (0): Linear(in_features=768, out_features=3072, bias=True)
          (1): GELU()
          (2): Linear(in_features=3072, out_features=768, bias=True)
        )
      )
      (norm1): LayerNorm()
      (norm2): LayerNorm()
      (drop_resid): Dropout(p=0.0, inplace=False)
    )
    (1): TransformerBlock(
      (att): MultiHeadAttention(
        (W_query): Linear(in_features=768,

In [37]:
inputs= tokenizer.encode("do you have time ??")
inputs= torch.tensor(inputs).unsqueeze(0)
print("inputs", inputs)
print("input dims", inputs.shape)

inputs tensor([[ 4598,   345,   423,   640, 19153]])
input dims torch.Size([1, 5])


In [41]:
with torch.no_grad():
    outputs= model(inputs)
print("outputs:\n", outputs)
print("op dims:" ,outputs.shape)

outputs:
 tensor([[[-1.5465,  0.9756],
         [-3.7792,  7.4043],
         [-2.3096,  6.4195],
         [-3.6631,  4.2262],
         [-4.0467,  4.9151]]])
op dims: torch.Size([1, 5, 2])


In [42]:
print("last output token:" , outputs[:, -1, :])

last output token: tensor([[-4.0467,  4.9151]])


In [54]:
def calc_accuracy_loader(data_loader, model, device, num_batches= None):
    model.eval()
    correct_predictions, num_examples= 0,0

    if num_batches is None:
        num_batches = len(data_loader)
    else :
        num_batches= min(num_batches, len(data_loader))

    for i , (input_batch, target_batch) in enumerate(data_loader):
        if i< num_batches:
            input_batch= input_batch.to(device)
            target_batch= target_batch.to(device)
            # print("hare" ,i, input_batch.shape, input_batch)
            with torch.no_grad():
                logits= model(input_batch)[:, -1, :]
                print("haribol", logits)
            predicted_labels = torch.argmax(logits, dim=-1)
            # print("krishna",i,predicted_labels.shape, predicted_labels)

            num_examples += predicted_labels.shape[0]
            correct_predictions +=(
                (predicted_labels == target_batch).sum().item()
            )
            # print("redhe",i, correct_predictions)
        else:
            break

    
    return correct_predictions / num_examples

In [55]:
device= torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

torch.manual_seed(123)
train_accuracy = calc_accuracy_loader(
    train_loader, model, device, num_batches=10
)

print(f"Training accuracy : {train_accuracy*100:.2f}%")

haribol tensor([[-2.3507,  2.6437],
        [-2.4689,  2.7582],
        [-2.3161,  2.7413],
        [-2.4349,  2.7347],
        [-2.3471,  2.7348],
        [-2.4429,  2.7201],
        [-2.4104,  2.8182],
        [-2.4334,  2.7510]])
haribol tensor([[-2.3874,  2.8109],
        [-2.4590,  2.7197],
        [-2.3133,  2.7725],
        [-2.4135,  2.6959],
        [-2.4366,  2.7930],
        [-2.4560,  2.7447],
        [-2.3810,  2.7074],
        [-2.4547,  2.7679]])
haribol tensor([[-2.4327,  2.8559],
        [-2.4020,  2.8596],
        [-2.4900,  2.7955],
        [-2.3863,  2.6997],
        [-2.3134,  2.7227],
        [-2.3506,  2.6729],
        [-2.4694,  2.7965],
        [-2.4771,  2.7097]])
haribol tensor([[-2.3716,  2.7096],
        [-2.5092,  2.8159],
        [-2.3230,  2.7485],
        [-2.4385,  2.7252],
        [-2.1310,  2.6975],
        [-2.3092,  2.7141],
        [-2.4015,  2.7743],
        [-2.3950,  2.6884]])
haribol tensor([[-2.2588,  2.6989],
        [-2.4250,  2.7171],
    

In [52]:
def calc_loss_batch(input_batch, target_batch, model, device):
    input_batch = input_batch.to(device)
    target_batch= target_batch.to(device)
    logits= model(input_batch)[:, -1, :]
    loss= torch.nn.functional.cross_entropy(logits, target_batch)
    return loss

In [56]:
def calc_loss_loader(data_loader, model, device, num_batches= None):
    total_loss=0
    if len(data_loader)==0:
        return float("nan")
    
    elif num_batches is None:
        num_batches = len(data_loader)
    else:
        num_batches= min(num_batches, len(data_loader))
    
    for i , (input_batch, target_batch ) in enumerate(data_loader):
        if i< num_batches:
            loss= calc_loss_batch(
                input_batch, target_batch,model, device
            )
            total_loss += loss.item()

        else:
            break
    
    return total_loss / num_batches

In [ ]:
# lets find the initial loss of each dataset
with torch.no_grad():
    train_loss= calc_loss_loader(
        train_loader, model, device, num_batches=5
    )
    print("train loss", train_loss)  #yes yes you can find for validation and test also\

train loss 2.325549602508545


In [62]:
def train_classifier_simple(
        model, train_loader, val_loader, optimizer, device,
        num_epochs, eval_freq, eval_iter):
    train_losses, val_losses, train_accs, val_accs = [], [], [], []   #1
    examples_seen, global_step = 0, -1

    for epoch in range(num_epochs):   #2
        model.train()                 #3

        for input_batch, target_batch in train_loader:
            optimizer.zero_grad()     #4

            loss = calc_loss_batch(
                input_batch, target_batch, model, device
            )

            loss.backward()           #5
            optimizer.step()          #6
            examples_seen += input_batch.shape[0]   #7
            global_step += 1

            if global_step % eval_freq == 0:        #8
                train_loss, val_loss = evaluate_model(
                    model, train_loader, val_loader, device, eval_iter
                )
                train_losses.append(train_loss)
                val_losses.append(val_loss)

                print(
                    f"Ep {epoch+1} (Step {global_step:06d}): "
                    f"Train loss {train_loss:.3f}, "
                    f"Val loss {val_loss:.3f}"
                )

        train_accuracy = calc_accuracy_loader(      #9
            train_loader, model, device, num_batches=eval_iter
        )

        val_accuracy = calc_accuracy_loader(
            val_loader, model, device, num_batches=eval_iter
        )

        print(f"Training accuracy: {train_accuracy*100:.2f}% | ", end="")
        print(f"Validation accuracy: {val_accuracy*100:.2f}%")

        train_accs.append(train_accuracy)
        val_accs.append(val_accuracy)

    return train_losses, val_losses, train_accs, val_accs, examples_seen

In [60]:
def evaluate_model(model, train_loader, val_loader, device, eval_iter):
    model.eval()
    with torch.no_grad():
        train_loss= calc_loss_loader(
            train_loader, model, device, num_batches= eval_iter
        )
        val_loss= calc_loss_loader(
            val_loader, model, device, num_batches= eval_iter
        )

    model.train()
    return train_loss, val_loss

In [63]:
import time
start_time= time.time()
torch.manual_seed(123)
optimizer= torch.optim.AdamW(model.parameters(), lr= 5e-5, weight_decay= 0.1)
num_epochs=5

train_losses, val_losses, train_accs, val_accs, examples_seen= \
    train_classifier_simple(
        model, train_loader, val_loader, optimizer, device,
        num_epochs= num_epochs, eval_freq=50,
        eval_iter=5
    )

end_time= time.time()
execution_time_minutes= (end_time - start_time)/60
print(f"training complete in {execution_time_minutes}")

Ep 1 (Step 000000): Train loss 2.153, Val loss 2.392
Ep 1 (Step 000050): Train loss 0.617, Val loss 0.637
Ep 1 (Step 000100): Train loss 0.523, Val loss 0.557
haribol tensor([[ 0.6315,  0.4142],
        [ 0.4163,  0.6638],
        [ 0.9974,  0.1842],
        [ 0.4806,  0.6430],
        [ 0.7462,  0.4111],
        [ 0.9333,  0.2625],
        [ 1.2382, -0.1150],
        [ 0.6085,  0.5267]])
haribol tensor([[ 0.5686,  0.5293],
        [ 1.8539, -0.5943],
        [ 0.3747,  0.7147],
        [ 0.6276,  0.5102],
        [ 1.5586, -0.3700],
        [ 0.6046,  0.5496],
        [ 0.6486,  0.4687],
        [ 1.3622, -0.2048]])
haribol tensor([[ 0.4665,  0.6443],
        [ 1.3769, -0.1646],
        [ 0.8416,  0.2516],
        [ 1.4001, -0.2192],
        [-0.0355,  0.9099],
        [ 1.1614, -0.0410],
        [ 0.4793,  0.5676],
        [ 1.4890, -0.3049]])
haribol tensor([[ 0.6105,  0.5106],
        [ 0.6274,  0.5126],
        [ 0.4669,  0.6398],
        [ 1.3899, -0.1978],
        [ 0.4415,  0.7

In [64]:
torch.save(model.state_dict(), "review_classifier.pth")

In [ ]:
# print(model.tok_emb)

Embedding(50257, 768)


In [72]:
# print(model)

In [ ]:
def classify_review(
    text, model, tokenizer, device, max_length=None,
    pad_token_id= 50256):
    model.eval()

    input_ids= tokenizer.encode(text)
    supported_context_length= model.pos_emb.weight.shape[1]
    
    input_ids= input_ids[:min(
        max_length, supported_context_length
    )]
    input_ids+= [pad_token_id] *(max_length - len(input_ids))

    input_tensor



In [73]:
s= [1,2,3]

In [74]:
s = s+ ["hari"]
s

[1, 2, 3, 'hari']